# deberta `{1e-5}` — context / epochs cell runner

**Set `WINDOW` and `EPOCHS` in cell 1. Nothing else changes between cells.**

Currently configured for the **control**: `WINDOW = 0`, `EPOCHS = 6`.

## Why this run exists

`deberta_ctx32` (context 32, 6 epochs) beat E08 on both domains — equi +0.0273
(t 5.51, 5+/0−), htfl +0.0231 (t 2.48, 5+/0−). But it changed **two** things at
once against E08: context on, *and* 5 epochs → 6. Six epochs is also a different
LR schedule — 1,722 optimizer steps instead of 1,435, so a longer warmup and a
gentler decay.

This cell holds epochs at 6 and turns context **off**, which makes the three
cells decompose cleanly:

| cell | context | epochs | status |
|---|:--:|:--:|---|
| E08 | 0 | 5 | done — `results/runs/t11/deberta_lr1e-05_e5_terms/` |
| **this run** | **0** | **6** | **the missing control** |
| deberta_ctx32 | 32 | 6 | done |

E08 → this run isolates the epoch/schedule change.
This run → ctx32 isolates context, at a fixed schedule.

Without it, neither number in `deberta_ctx32` can be attributed.

Budget ~45 min — no context to tokenize, so faster than ctx32 despite the extra
epoch. Sidebar: **GPU T4 x2**, **Internet On**.


In [ ]:
# 1. THE TWO KNOBS, then Kaggle guard, GPU, and the command helper.
WINDOW = 0      # FLERT context tokens per side. 0 = sentence-level baseline.
EPOCHS = 6      # also sets the LR schedule horizon -- these are not separable

import os, sys, socket, subprocess, pathlib, shutil, json

if not pathlib.Path('/kaggle').is_dir():
    raise SystemExit('This notebook is for Kaggle.')
try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError as e:
    raise SystemExit(f'No internet ({e}). Sidebar -> Session options -> Internet -> On.')

import torch
assert torch.cuda.is_available(), 'No GPU. Sidebar -> Session options -> Accelerator -> GPU.'
print(torch.cuda.get_device_name(0), '|', torch.__version__, '| cuda', torch.version.cuda)

WORK = pathlib.Path('/kaggle/working')
REPO = pathlib.Path('/tmp/ate-acter')
# the group names BOTH knobs: deberta_ctx32 holds a 6-epoch cell under a name
# that says only the window, which is how the confound got in
GROUP = f'context/deberta_ctx{WINDOW}_e{EPOCHS}'
SRC = REPO / 'results/runs' / GROUP
print(f'\ncell: context window {WINDOW}, {EPOCHS} epochs -> {GROUP}')

def run(*args, cwd=None):
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=cwd, env=env)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {" ".join(str(a) for a in args)}')

In [ ]:
# 2. Clone the project and the corpus into /tmp.
shutil.rmtree(REPO, ignore_errors=True)
run('git', 'clone', '-q', 'https://github.com/ahmedwaleedaref/ATE-ACTER.git', REPO)
run('git', 'clone', '-q', 'https://github.com/AylaRT/ACTER.git', REPO / 'data/raw/ACTER')
run('git', 'checkout', '-q', 'f05b09e985cad37eeaa8daa8b3f383197aa5324e',
    cwd=REPO / 'data/raw/ACTER')

for path, needle, why in (
    ('src/models/run_train.py', '--context-window',       'the flag is absent'),
    ('src/data/dataset.py',     'context_window: int = 0', 'build_examples cannot take a window'),
    ('src/data/dataset.py',     'recover_sentence_labels', 'recovery would not offset past context'),
    ('src/models/run_train.py', '--dump-terms',           'no term list for the breakdown'),
):
    assert needle in (REPO / path).read_text(), f'clone predates {needle!r} in {path}: {why}'
assert (REPO / 'data/raw/ACTER/en/htfl/annotated').is_dir(), 'ACTER checkout looks wrong'
print(subprocess.run(['git','log','--oneline','-1'], cwd=REPO,
                     capture_output=True, text=True).stdout)

In [ ]:
# 3. Pinned installs.
run(sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==5.16.1', 'tokenizers==0.23.1', 'safetensors==0.8.0',
    'huggingface_hub==1.29.0', 'sentencepiece==0.2.2', 'protobuf==7.36.0',
    'PyYAML==6.0.3', 'pytest==8.3.2')

def _pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a],
                          capture_output=True, text=True).returncode == 0
HAVE_SEQEVAL = (_pip('--no-build-isolation', 'seqeval==1.2.2')
                or (_pip('setuptools<81', 'wheel')
                    and _pip('--no-build-isolation', 'seqeval==1.2.2')))
print('seqeval:', 'installed' if HAVE_SEQEVAL else 'UNAVAILABLE -- training unaffected')

In [ ]:
# 4. Tests, then the context gate -- only meaningful when WINDOW > 0.
import importlib
for m in ('torch', 'transformers', 'tokenizers', 'numpy'):
    print(f'{m:14} {importlib.import_module(m).__version__}')
skip = [] if HAVE_SEQEVAL else ['--ignore=tests/test_seqeval_agreement.py']
run(sys.executable, '-m', 'pytest', 'tests/', '-q', *skip, cwd=REPO)

if WINDOW == 0:
    print('\nWINDOW=0: context gate skipped -- it would compare the baseline to itself.')
else:
    gate = f'''
import dataclasses
from src.data.dataset import (ATEDataset, build_dataloader, build_examples,
                              get_tokenizer, load_train_config)
from src.data.align import positive_rate, IGNORE_INDEX
from src.statistics.loading import load_config
cfg = dataclasses.replace(load_train_config(), model_name="microsoft/deberta-v3-base")
tok, dcfg = get_tokenizer(cfg), load_config()
for dom, filt in (("corp", None), ("equi", None), ("wind", 2), ("htfl", None)):
    rates = {{}}
    for W in (0, {WINDOW}):
        ex = build_examples(dom, tokenizer=tok, truncation=True, max_length=cfg.max_length,
                            filter_max_tokens=filt, context_window=W, data_cfg=dcfg)
        ld = build_dataloader(ATEDataset(ex), tokenizer=tok, batch_size=32,
                              shuffle=False, length_grouped=False)
        rates[W] = positive_rate(ld)[0]
    leaks = sum(1 for e in ex for w, l in zip(e.word_ids, e.labels)
                if w is not None and not (e.n_left <= w < e.n_left + len(e.tokens))
                and l != IGNORE_INDEX)
    assert abs(rates[0] - rates[{WINDOW}]) < 1e-12, f"{{dom}}: rate moved {{rates}}"
    assert leaks == 0, f"{{dom}}: {{leaks}} context labels reached the loss"
    print(f"  {{dom:6}} rate {{rates[0]:.6f}} == {{rates[{WINDOW}]:.6f}}, 0 leaks")
print("GATE PASSED")
'''
    print('\ncontext gate:')
    run(sys.executable, '-c', gate, cwd=REPO)

In [ ]:
# 5. Seed 42 alone. Inspect before spending the rest.
def train(seed):
    run(sys.executable, '-m', 'src.models.run_train',
        '--model', 'microsoft/deberta-v3-base',
        '--lr', '1e-5', '--epochs', EPOCHS, '--context-window', WINDOW,
        '--group', GROUP, '--seed', seed, '--dump-terms',
        '--reason', f'context {WINDOW} / {EPOCHS} epochs -- decomposing E08 vs deberta_ctx32',
        cwd=REPO)
    r = json.loads((SRC / f'seed_{seed}.json').read_text())
    t = r['test']
    # a silently-ignored flag would produce a cell labelled as something it is not
    assert r['config']['context_window'] == WINDOW, 'context_window did not reach the run'
    assert r['config']['num_epochs'] == EPOCHS, 'num_epochs did not reach the run'
    assert r['test_term_list']['n_terms'] == t['n_pred_types']
    print(f"    SEED {seed}: best_epoch={r['best_epoch']} equi={r['best_equi_f1']:.4f} "
          f"htfl={t['list_ann_f1']:.4f} span_f1={t['span_f1']:.4f} "
          f"types={t['n_pred_types']} | {r['wall_time_sec']}s")
    return r

def persist():
    dest = WORK / GROUP
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(SRC, dest)
    print(f'  -> copied to {dest}')

print(f'########## seed 42 | ctx {WINDOW}, {EPOCHS} epochs ##########')
train(42)
persist()
print('\nreference cells, 5-seed means:')
print('  E08          ctx 0,  5ep : equi 0.5592 +/- 0.0090  htfl 0.5782 +/- 0.0230')
print('  deberta_ctx32 ctx 32, 6ep : equi 0.5866 +/- 0.0031  htfl 0.6013 +/- 0.0058')

In [ ]:
# 6. Seeds 43-46. Output re-copied after each.
for s in (43, 44, 45, 46):
    print(f'\n########## seed {s} ##########')
    train(s)
    persist()

In [ ]:
# 7. The decomposition. Paired on the seed, df=4: 2.132 suggestive, 2.776 serious.
import statistics as st
S = (42, 43, 44, 45, 46)
runs = {}
for p in sorted((WORK / GROUP).glob('seed_*.json')):
    r = json.loads(p.read_text())
    runs[r['seed']] = {'equi': r['best_equi_f1'], 'htfl': r['test']['list_ann_f1']}

E08  = {'equi': {42:0.5673, 43:0.5442, 44:0.5581, 45:0.5627, 46:0.5639},
        'htfl': {42:0.6046, 43:0.5783, 44:0.5652, 45:0.5474, 46:0.5953}}
CTX32 = {'equi': {42:0.5870, 43:0.5892, 44:0.5893, 45:0.5859, 46:0.5817},
         'htfl': {42:0.6096, 43:0.5933, 44:0.6010, 45:0.6006, 46:0.6020}}

def paired(a, b, name):
    d = [a[s] - b[s] for s in S]
    m, sd = st.mean(d), st.stdev(d)
    t = m / (sd / len(d) ** 0.5) if sd else float('inf')
    print(f'  {name:34} {m:+.4f}  s_d {sd:.4f}  t {t:5.2f}  '
          f'{sum(x > 0 for x in d)}+/{sum(x < 0 for x in d)}-')

if len(runs) == 5:
    for dom in ('equi', 'htfl'):
        this = {s: runs[s][dom] for s in S}
        print(f'\n{dom}:  this cell {st.mean(this.values()):.4f} '
              f'+/- {st.stdev(this.values()):.4f}')
        paired(this, E08[dom],   'EPOCHS alone (this - E08)')
        paired(CTX32[dom], this, 'CONTEXT alone (ctx32 - this)')
        paired(CTX32[dom], E08[dom], 'both together (ctx32 - E08)')
    print('\nThe two isolated effects should roughly sum to the combined one.')
    print('If they do not, epochs and context interact and neither is additive.')
    print('Report equi and htfl SEPARATELY -- different fill rates, which is the point.')
else:
    print(f'{len(runs)} of 5 seeds -- rerun cell 6.')